## Trạng thái bản lưu trên GitHub

Đề xuất ban đầu, giữ để đọc tiến trình; được thay bởi 06/07, không dùng để xuất model hiện tại.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import copy
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    DataLoader,
    Dataset,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)


BASE_SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(BASE_SEED)


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )
else:
    raise RuntimeError(
        "Notebook 04 cần bật GPU T4."
    )

In [ ]:
def find_kaggle_file(filename):
    """Tìm file trong working hoặc notebook input."""
    working_path = (
        Path("/kaggle/working") / filename
    )

    if working_path.exists():
        return working_path

    candidates = sorted(
        Path("/kaggle/input").rglob(filename)
    )

    if not candidates:
        raise FileNotFoundError(
            f"Không tìm thấy {filename}"
        )

    if len(candidates) > 1:
        print(
            "Tìm thấy nhiều file, đang dùng:",
            candidates[0],
        )

    return candidates[0]


FEATURE_PATH = find_kaggle_file(
    "gmdcsa24_features_32.npz"
)


with np.load(FEATURE_PATH) as data:
    skeleton_features = data[
        "skeleton_features"
    ].astype(np.float32)

    motion_features = data[
        "motion_features"
    ].astype(np.float32)

    global_features = data[
        "global_features"
    ].astype(np.float32)

    body_scales = data[
        "body_scales"
    ].astype(np.float32)

    labels = data[
        "labels"
    ].astype(np.int64)

    subjects = data[
        "subjects"
    ].astype(np.int64)

    video_ids = data[
        "video_ids"
    ].astype("U")


num_samples = len(video_ids)
num_frames = skeleton_features.shape[1]
num_joints = skeleton_features.shape[2]


# x, y, visibility
pose_input = skeleton_features[
    ...,
    [0, 1, 3],
]


# velocity_x, velocity_y
velocity_input = motion_features[
    ...,
    :2,
]


# Góc nghiêng thân
torso_angle = global_features[
    ...,
    9,
]


# Vị trí tâm hông x, y
hip_position = global_features[
    ...,
    :2,
]


# Dịch chuyển tâm hông so với frame đầu
safe_body_scales = np.maximum(
    body_scales,
    1e-6,
)

hip_displacement = (
    hip_position - hip_position[:, :1]
) / safe_body_scales[:, None, None]


# Lặp đặc trưng toàn thân trên 33 khớp
torso_channel = np.broadcast_to(
    torso_angle[:, :, None, None],
    (
        num_samples,
        num_frames,
        num_joints,
        1,
    ),
)

hip_x_channel = np.broadcast_to(
    hip_displacement[
        :, :, 0, None, None
    ],
    (
        num_samples,
        num_frames,
        num_joints,
        1,
    ),
)

hip_y_channel = np.broadcast_to(
    hip_displacement[
        :, :, 1, None, None
    ],
    (
        num_samples,
        num_frames,
        num_joints,
        1,
    ),
)


# Ghép thành 8 kênh
motion_enhanced_input = np.concatenate(
    [
        pose_input,
        velocity_input,
        torso_channel,
        hip_x_channel,
        hip_y_channel,
    ],
    axis=-1,
).astype(np.float32)


# N,T,V,C → N,C,T,V
motion_enhanced_input = np.transpose(
    motion_enhanced_input,
    (0, 3, 1, 2),
).astype(np.float32)


CHANNEL_NAMES = [
    "relative_x",
    "relative_y",
    "visibility",
    "velocity_x",
    "velocity_y",
    "torso_angle",
    "hip_displacement_x",
    "hip_displacement_y",
]


assert motion_enhanced_input.shape == (
    160,
    8,
    32,
    33,
)

assert np.isfinite(
    motion_enhanced_input
).all()


all_subjects = sorted(
    np.unique(subjects).tolist()
)


print("Feature path:", FEATURE_PATH)
print(
    "Motion-enhanced input:",
    motion_enhanced_input.shape,
)
print("Channels:", CHANNEL_NAMES)
print("Subjects:", all_subjects)
print(
    "Finite:",
    np.isfinite(
        motion_enhanced_input
    ).all(),
)

In [ ]:
NUM_JOINTS = 33


POSE_EDGES = [
    # Khuôn mặt
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 7),

    (0, 4),
    (4, 5),
    (5, 6),
    (6, 8),

    (0, 9),
    (0, 10),
    (9, 10),

    # Đầu và vai
    (0, 11),
    (0, 12),
    (11, 12),

    # Tay trái
    (11, 13),
    (13, 15),
    (15, 17),
    (15, 19),
    (15, 21),
    (17, 19),

    # Tay phải
    (12, 14),
    (14, 16),
    (16, 18),
    (16, 20),
    (16, 22),
    (18, 20),

    # Thân
    (11, 23),
    (12, 24),
    (23, 24),

    # Chân trái
    (23, 25),
    (25, 27),
    (27, 29),
    (29, 31),
    (27, 31),

    # Chân phải
    (24, 26),
    (26, 28),
    (28, 30),
    (30, 32),
    (28, 32),
]


adjacency = np.eye(
    NUM_JOINTS,
    dtype=np.float32,
)


for start_joint, end_joint in POSE_EDGES:
    adjacency[
        start_joint,
        end_joint,
    ] = 1.0

    adjacency[
        end_joint,
        start_joint,
    ] = 1.0


# D^(-1/2) A D^(-1/2)
degree = adjacency.sum(axis=1)

inverse_sqrt_degree = np.power(
    np.maximum(degree, 1e-6),
    -0.5,
)

degree_matrix = np.diag(
    inverse_sqrt_degree
)

normalized_adjacency = (
    degree_matrix
    @ adjacency
    @ degree_matrix
).astype(np.float32)


ADJACENCY = torch.tensor(
    normalized_adjacency,
    dtype=torch.float32,
)


# Các cặp khớp trái-phải
LEFT_RIGHT_PAIRS = [
    (1, 4),
    (2, 5),
    (3, 6),
    (7, 8),
    (9, 10),

    (11, 12),
    (13, 14),
    (15, 16),
    (17, 18),
    (19, 20),
    (21, 22),

    (23, 24),
    (25, 26),
    (27, 28),
    (29, 30),
    (31, 32),
]


# Các kênh phải đổi dấu khi lật ngang
HORIZONTAL_SIGN_CHANNELS = [
    0,  # relative_x
    3,  # velocity_x
    5,  # torso_angle
    6,  # hip_displacement_x
]


def temporal_crop_and_resize(
    sample,
    minimum_ratio=0.85,
):
    """Cắt chuỗi rồi nội suy lại 32 frame."""
    channels, frames, joints = (
        sample.shape
    )

    minimum_length = max(
        2,
        int(round(frames * minimum_ratio)),
    )

    crop_length = random.randint(
        minimum_length,
        frames,
    )

    if crop_length == frames:
        return sample

    start_index = random.randint(
        0,
        frames - crop_length,
    )

    cropped = sample[
        :,
        start_index:
        start_index + crop_length,
        :,
    ]

    # C,T,V → 1,C*V,T
    reshaped = (
        cropped
        .permute(0, 2, 1)
        .contiguous()
        .reshape(
            1,
            channels * joints,
            crop_length,
        )
    )

    resized = F.interpolate(
        reshaped,
        size=frames,
        mode="linear",
        align_corners=False,
    )

    # 1,C*V,T → C,T,V
    resized = (
        resized
        .reshape(
            channels,
            joints,
            frames,
        )
        .permute(0, 2, 1)
        .contiguous()
    )

    return resized


def horizontal_flip(sample):
    """Lật ngang và hoán đổi khớp trái-phải."""
    sample = sample.clone()

    sample[
        HORIZONTAL_SIGN_CHANNELS
    ] *= -1.0

    for left_joint, right_joint in (
        LEFT_RIGHT_PAIRS
    ):
        left_values = sample[
            :, :, left_joint
        ].clone()

        sample[
            :, :, left_joint
        ] = sample[
            :, :, right_joint
        ]

        sample[
            :, :, right_joint
        ] = left_values

    return sample


def joint_dropout(
    sample,
    maximum_dropped_joints=2,
):
    """Ẩn ngẫu nhiên một vài node."""
    sample = sample.clone()

    dropped_count = random.randint(
        1,
        maximum_dropped_joints,
    )

    dropped_joints = random.sample(
        range(NUM_JOINTS),
        k=dropped_count,
    )

    sample[
        :, :, dropped_joints
    ] = 0.0

    return sample


class MotionEnhancedDataset(Dataset):
    def __init__(
        self,
        indices,
        augment=False,
    ):
        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        sample_index = int(
            self.indices[item]
        )

        sample = torch.from_numpy(
            motion_enhanced_input[
                sample_index
            ]
        ).clone()

        target = torch.tensor(
            labels[sample_index],
            dtype=torch.long,
        )

        if self.augment:
            # Temporal crop
            if random.random() < 0.5:
                sample = (
                    temporal_crop_and_resize(
                        sample
                    )
                )

            # Horizontal flip
            if random.random() < 0.5:
                sample = horizontal_flip(
                    sample
                )

            # Nhiễu nhỏ trên tọa độ x, y
            if random.random() < 0.5:
                coordinate_noise = (
                    torch.randn_like(
                        sample[0:2]
                    )
                    * 0.01
                )

                sample[0:2] += (
                    coordinate_noise
                )

            # Joint dropout
            if random.random() < 0.3:
                sample = joint_dropout(
                    sample
                )

        return sample, target


print("Adjacency:", ADJACENCY.shape)
print("Số cạnh:", len(POSE_EDGES))
print(
    "Đối xứng:",
    torch.allclose(
        ADJACENCY,
        ADJACENCY.T,
    ),
)


# Kiểm tra Dataset
test_dataset = MotionEnhancedDataset(
    indices=[0],
    augment=True,
)

test_sample, test_target = (
    test_dataset[0]
)

print("Sample:", test_sample.shape)
print("Label:", test_target.item())
print(
    "Finite:",
    torch.isfinite(test_sample).all(),
)

In [ ]:
class SpatialGraphConv(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
    ):
        super().__init__()

        self.projection = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=1,
        )

    def forward(
        self,
        features,
        adjacency_matrix,
    ):
        features = torch.einsum(
            "nctv,vw->nctw",
            features,
            adjacency_matrix,
        )

        return self.projection(features)


class STGCNBlock(nn.Module):
    def __init__(
        self,
        input_channels,
        output_channels,
        temporal_stride=1,
        dropout=0.3,
        residual=True,
    ):
        super().__init__()

        self.spatial_graph_conv = (
            SpatialGraphConv(
                input_channels,
                output_channels,
            )
        )

        self.temporal_conv = nn.Sequential(
            nn.BatchNorm2d(
                output_channels
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                output_channels,
                output_channels,
                kernel_size=(9, 1),
                stride=(
                    temporal_stride,
                    1,
                ),
                padding=(4, 0),
            ),

            nn.BatchNorm2d(
                output_channels
            ),

            nn.Dropout(dropout),
        )

        if not residual:
            self.residual = None

        elif (
            input_channels == output_channels
            and temporal_stride == 1
        ):
            self.residual = nn.Identity()

        else:
            self.residual = nn.Sequential(
                nn.Conv2d(
                    input_channels,
                    output_channels,
                    kernel_size=1,
                    stride=(
                        temporal_stride,
                        1,
                    ),
                ),
                nn.BatchNorm2d(
                    output_channels
                ),
            )

        self.activation = nn.ReLU(
            inplace=True
        )

    def forward(
        self,
        features,
        adjacency_matrix,
    ):
        residual = (
            0
            if self.residual is None
            else self.residual(features)
        )

        features = self.spatial_graph_conv(
            features,
            adjacency_matrix,
        )

        features = self.temporal_conv(
            features
        )

        return self.activation(
            features + residual
        )


class TemporalAttention(nn.Module):
    def __init__(
        self,
        channels,
        hidden_channels=64,
    ):
        super().__init__()

        self.attention_network = nn.Sequential(
            nn.Conv1d(
                channels,
                hidden_channels,
                kernel_size=1,
            ),
            nn.Tanh(),
            nn.Conv1d(
                hidden_channels,
                1,
                kernel_size=1,
            ),
        )

    def forward(self, temporal_features):
        # temporal_features: N,C,T
        attention_logits = (
            self.attention_network(
                temporal_features
            )
            .squeeze(1)
        )

        attention_weights = torch.softmax(
            attention_logits,
            dim=-1,
        )

        pooled_features = torch.sum(
            temporal_features
            * attention_weights.unsqueeze(1),
            dim=-1,
        )

        return (
            pooled_features,
            attention_weights,
        )


class MESTGCNTA(nn.Module):
    def __init__(
        self,
        adjacency_matrix,
        input_channels=8,
        num_classes=2,
        dropout=0.3,
    ):
        super().__init__()

        self.register_buffer(
            "adjacency",
            adjacency_matrix.clone(),
        )

        self.input_batch_norm = nn.BatchNorm1d(
            input_channels * NUM_JOINTS
        )

        self.blocks = nn.ModuleList([
            STGCNBlock(
                input_channels,
                64,
                dropout=dropout,
                residual=False,
            ),
            STGCNBlock(
                64,
                64,
                dropout=dropout,
            ),
            STGCNBlock(
                64,
                128,
                temporal_stride=2,
                dropout=dropout,
            ),
            STGCNBlock(
                128,
                128,
                dropout=dropout,
            ),
        ])

        self.edge_importance = (
            nn.ParameterList([
                nn.Parameter(
                    torch.ones_like(
                        adjacency_matrix
                    )
                )
                for _ in self.blocks
            ])
        )

        self.temporal_attention = (
            TemporalAttention(
                channels=128,
                hidden_channels=64,
            )
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(
                128,
                num_classes,
            ),
        )

    def forward(self, features):
        # Input: N,C,T,V
        batch_size, channels, frames, joints = (
            features.shape
        )

        # N,C,T,V → N,V*C,T
        features = (
            features
            .permute(0, 3, 1, 2)
            .contiguous()
            .view(
                batch_size,
                joints * channels,
                frames,
            )
        )

        features = self.input_batch_norm(
            features
        )

        # N,V*C,T → N,C,T,V
        features = (
            features
            .view(
                batch_size,
                joints,
                channels,
                frames,
            )
            .permute(0, 2, 3, 1)
            .contiguous()
        )

        for block, importance in zip(
            self.blocks,
            self.edge_importance,
        ):
            features = block(
                features,
                self.adjacency * importance,
            )

        # Average pooling qua 33 joints
        temporal_features = features.mean(
            dim=3
        )

        (
            video_features,
            attention_weights,
        ) = self.temporal_attention(
            temporal_features
        )

        logits = self.classifier(
            video_features
        )

        return logits, attention_weights


set_seed(BASE_SEED)

test_model = MESTGCNTA(
    ADJACENCY
).to(DEVICE)


test_input = torch.from_numpy(
    motion_enhanced_input[:2]
).to(DEVICE)


with torch.no_grad():
    test_logits, test_attention = (
        test_model(test_input)
    )


parameter_count = sum(
    parameter.numel()
    for parameter in test_model.parameters()
)


print("Input:", test_input.shape)
print("Logits:", test_logits.shape)
print(
    "Attention:",
    test_attention.shape,
)
print(
    "Tổng attention mẫu 1:",
    test_attention[0].sum().item(),
)
print(
    "Số tham số:",
    f"{parameter_count:,}",
)


del test_model
del test_input

torch.cuda.empty_cache()

In [ ]:
BATCH_SIZE = 16


def create_loader(
    indices,
    shuffle,
    augment,
    seed,
):
    dataset = MotionEnhancedDataset(
        indices=indices,
        augment=augment,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        generator=generator,
    )


def calculate_class_weights(
    train_indices,
):
    """Chỉ tính từ tập train."""
    class_counts = np.bincount(
        labels[train_indices],
        minlength=2,
    ).astype(np.float32)

    if np.any(class_counts == 0):
        raise ValueError(
            "Tập train thiếu một lớp."
        )

    class_weights = (
        len(train_indices)
        / (2.0 * class_counts)
    )

    return torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE,
    )


def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    false_alarm_rate = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "sensitivity": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "false_alarm_rate": false_alarm_rate,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_one_epoch(
    model,
    data_loader,
    optimizer,
    criterion,
):
    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, targets in data_loader:
        inputs = inputs.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits, _ = model(inputs)

        loss = criterion(
            logits,
            targets,
        )

        loss.backward()

        # Hạn chế gradient quá lớn
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        current_batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * current_batch_size
        )

        total_samples += (
            current_batch_size
        )

    return total_loss / total_samples


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    criterion,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0

    true_labels = []
    predicted_labels = []
    fall_probabilities = []
    attention_values = []

    for inputs, targets in data_loader:
        inputs = inputs.to(
            DEVICE,
            non_blocking=True,
        )

        targets = targets.to(
            DEVICE,
            non_blocking=True,
        )

        (
            logits,
            attention_weights,
        ) = model(inputs)

        loss = criterion(
            logits,
            targets,
        )

        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        predictions = logits.argmax(
            dim=1
        )

        current_batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * current_batch_size
        )

        total_samples += (
            current_batch_size
        )

        true_labels.extend(
            targets.cpu().numpy()
        )

        predicted_labels.extend(
            predictions.cpu().numpy()
        )

        fall_probabilities.extend(
            probabilities[:, 1]
            .cpu()
            .numpy()
        )

        attention_values.extend(
            attention_weights
            .cpu()
            .numpy()
        )

    true_labels = np.asarray(
        true_labels,
        dtype=np.int64,
    )

    predicted_labels = np.asarray(
        predicted_labels,
        dtype=np.int64,
    )

    fall_probabilities = np.asarray(
        fall_probabilities,
        dtype=np.float32,
    )

    attention_values = np.asarray(
        attention_values,
        dtype=np.float32,
    )

    metrics = calculate_metrics(
        true_labels,
        predicted_labels,
    )

    return (
        total_loss / total_samples,
        metrics,
        true_labels,
        predicted_labels,
        fall_probabilities,
        attention_values,
    )


print(
    "Class weights mẫu:",
    calculate_class_weights(
        np.arange(100)
    ).detach().cpu().numpy(),
)

In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 120
MIN_EPOCHS = 20
PATIENCE = 20

DROPOUT = 0.3


def train_with_early_stopping(
    train_indices,
    val_indices,
    seed,
):
    set_seed(seed)

    model = MESTGCNTA(
        ADJACENCY,
        dropout=DROPOUT,
    ).to(DEVICE)

    train_loader = create_loader(
        train_indices,
        shuffle=True,
        augment=True,
        seed=seed,
    )

    val_loader = create_loader(
        val_indices,
        shuffle=False,
        augment=False,
        seed=seed,
    )

    class_weights = (
        calculate_class_weights(
            train_indices
        )
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_state = None
    best_epoch = 0
    best_metrics = None

    best_score = (
        -np.inf,
        -np.inf,
        -np.inf,
    )

    waiting_epochs = 0
    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
        )

        (
            val_loss,
            val_metrics,
            _,
            _,
            _,
            _,
        ) = evaluate_model(
            model,
            val_loader,
            criterion,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **val_metrics,
        })

        current_score = (
            val_metrics[
                "balanced_accuracy"
            ],
            val_metrics["f1"],
            -val_loss,
        )

        if current_score > best_score:
            best_score = current_score
            best_epoch = epoch
            best_metrics = (
                val_metrics.copy()
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

            waiting_epochs = 0

        else:
            waiting_epochs += 1

        if (
            epoch >= MIN_EPOCHS
            and waiting_epochs >= PATIENCE
        ):
            break

    model.load_state_dict(
        best_state
    )

    return (
        model,
        best_epoch,
        best_metrics,
        pd.DataFrame(history),
        class_weights.detach().cpu().numpy(),
    )


def train_fixed_epochs(
    train_indices,
    number_of_epochs,
    seed,
):
    set_seed(seed)

    model = MESTGCNTA(
        ADJACENCY,
        dropout=DROPOUT,
    ).to(DEVICE)

    train_loader = create_loader(
        train_indices,
        shuffle=True,
        augment=True,
        seed=seed,
    )

    class_weights = (
        calculate_class_weights(
            train_indices
        )
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    history = []

    for epoch in range(
        1,
        number_of_epochs + 1,
    ):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
        })

    return (
        model,
        pd.DataFrame(history),
        class_weights.detach().cpu().numpy(),
    )


print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Dropout:", DROPOUT)
print("Max epochs:", MAX_EPOCHS)
print("Patience:", PATIENCE)

In [ ]:
OUTPUT_DIR = Path(
    "/kaggle/working/"
    "me_stgcn_ta_results"
)

MODEL_DIR = Path(
    "/kaggle/working/"
    "me_stgcn_ta_models"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


inner_result_records = []
outer_result_records = []
prediction_records = []

# Mỗi video sẽ có 16 trọng số attention
all_attention_weights = np.zeros(
    (
        len(video_ids),
        16,
    ),
    dtype=np.float32,
)

completed_test_subjects = []


experiment_start_time = time.time()


for test_subject in all_subjects:
    print("\n" + "=" * 65)
    print(
        f"OUTER TEST SUBJECT: S{test_subject}"
    )
    print("=" * 65)

    outer_start_time = time.time()

    development_subjects = [
        subject
        for subject in all_subjects
        if subject != test_subject
    ]

    inner_best_epochs = []

    # Ba inner folds
    for val_subject in development_subjects:
        train_subjects = [
            subject
            for subject in development_subjects
            if subject != val_subject
        ]

        train_indices = np.where(
            np.isin(
                subjects,
                train_subjects,
            )
        )[0]

        val_indices = np.where(
            subjects == val_subject
        )[0]

        current_seed = (
            BASE_SEED
            + test_subject * 100
            + val_subject
        )

        (
            inner_model,
            best_epoch,
            best_metrics,
            inner_history,
            inner_class_weights,
        ) = train_with_early_stopping(
            train_indices,
            val_indices,
            seed=current_seed,
        )

        inner_best_epochs.append(
            best_epoch
        )

        inner_result_records.append({
            "test_subject": test_subject,
            "val_subject": val_subject,
            "train_subjects": ",".join(
                map(str, train_subjects)
            ),
            "train_samples": len(
                train_indices
            ),
            "val_samples": len(
                val_indices
            ),
            "class_weight_adl": float(
                inner_class_weights[0]
            ),
            "class_weight_fall": float(
                inner_class_weights[1]
            ),
            "best_epoch": best_epoch,
            **best_metrics,
        })

        inner_history.to_csv(
            OUTPUT_DIR
            / (
                f"history_outer_{test_subject}"
                f"_val_{val_subject}.csv"
            ),
            index=False,
        )

        print(
            f"Inner val S{val_subject}"
            f" | train {train_subjects}"
            f" | best epoch {best_epoch}"
            f" | balanced accuracy "
            f"{best_metrics['balanced_accuracy']:.4f}"
            f" | F1 "
            f"{best_metrics['f1']:.4f}"
        )

        del inner_model

        torch.cuda.empty_cache()

    # Epoch cuối không lấy từ outer test
    final_epochs = int(
        round(
            np.median(
                inner_best_epochs
            )
        )
    )

    final_epochs = max(
        final_epochs,
        1,
    )

    print(
        "Inner best epochs:",
        inner_best_epochs,
    )

    print(
        "Final training epochs:",
        final_epochs,
    )

    # Train lại bằng toàn bộ ba subject
    final_train_indices = np.where(
        subjects != test_subject
    )[0]

    test_indices = np.where(
        subjects == test_subject
    )[0]

    final_seed = (
        BASE_SEED
        + 1000
        + test_subject
    )

    (
        final_model,
        final_history,
        final_class_weights,
    ) = train_fixed_epochs(
        final_train_indices,
        final_epochs,
        seed=final_seed,
    )

    test_loader = create_loader(
        test_indices,
        shuffle=False,
        augment=False,
        seed=final_seed,
    )

    final_criterion = (
        nn.CrossEntropyLoss(
            weight=torch.tensor(
                final_class_weights,
                dtype=torch.float32,
                device=DEVICE,
            )
        )
    )

    (
        test_loss,
        test_metrics,
        test_true,
        test_prediction,
        fall_probabilities,
        test_attention_weights,
    ) = evaluate_model(
        final_model,
        test_loader,
        final_criterion,
    )

    all_attention_weights[
        test_indices
    ] = test_attention_weights

    outer_elapsed_seconds = (
        time.time() - outer_start_time
    )

    outer_result_records.append({
        "test_subject": test_subject,
        "train_subjects": ",".join(
            map(
                str,
                development_subjects,
            )
        ),
        "inner_best_epochs": ",".join(
            map(
                str,
                inner_best_epochs,
            )
        ),
        "final_epochs": final_epochs,
        "train_samples": len(
            final_train_indices
        ),
        "test_samples": len(
            test_indices
        ),
        "class_weight_adl": float(
            final_class_weights[0]
        ),
        "class_weight_fall": float(
            final_class_weights[1]
        ),
        "test_loss": test_loss,
        "elapsed_seconds": (
            outer_elapsed_seconds
        ),
        **test_metrics,
    })

    for local_index, sample_index in enumerate(
        test_indices
    ):
        prediction_records.append({
            "sample_index": int(
                sample_index
            ),
            "video_id": video_ids[
                sample_index
            ],
            "subject": int(
                subjects[sample_index]
            ),
            "true_label": int(
                test_true[local_index]
            ),
            "predicted_label": int(
                test_prediction[
                    local_index
                ]
            ),
            "fall_probability": float(
                fall_probabilities[
                    local_index
                ]
            ),
            "correct": bool(
                test_true[local_index]
                == test_prediction[
                    local_index
                ]
            ),
        })

    final_history.to_csv(
        OUTPUT_DIR
        / (
            f"final_history_test_"
            f"subject_{test_subject}.csv"
        ),
        index=False,
    )

    cpu_state_dict = {
        name: value.detach().cpu()
        for name, value
        in final_model.state_dict().items()
    }

    model_path = (
        MODEL_DIR
        / (
            f"me_stgcn_ta_test_"
            f"subject_{test_subject}.pt"
        )
    )

    torch.save(
        {
            "model_state_dict": (
                cpu_state_dict
            ),
            "test_subject": test_subject,
            "final_epochs": final_epochs,
            "input_channels": 8,
            "channel_names": (
                CHANNEL_NAMES
            ),
            "num_joints": 33,
            "num_frames": 32,
            "attention_frames": 16,
            "dropout": DROPOUT,
            "learning_rate": (
                LEARNING_RATE
            ),
            "weight_decay": (
                WEIGHT_DECAY
            ),
            "class_weights": (
                final_class_weights
            ),
            "adjacency": ADJACENCY,
        },
        model_path,
    )

    completed_test_subjects.append(
        test_subject
    )

    # Lưu checkpoint kết quả sau mỗi subject
    pd.DataFrame(
        inner_result_records
    ).to_csv(
        OUTPUT_DIR
        / "inner_results.csv",
        index=False,
    )

    pd.DataFrame(
        outer_result_records
    ).to_csv(
        OUTPUT_DIR
        / "outer_results.csv",
        index=False,
    )

    pd.DataFrame(
        prediction_records
    ).to_csv(
        OUTPUT_DIR
        / "predictions.csv",
        index=False,
    )

    np.savez_compressed(
        OUTPUT_DIR
        / "attention_weights.npz",
        attention_weights=(
            all_attention_weights
        ),
        labels=labels,
        subjects=subjects,
        video_ids=video_ids,
        completed_test_subjects=np.asarray(
            completed_test_subjects,
            dtype=np.int64,
        ),
    )

    print(
        f"Test S{test_subject}"
        f" | accuracy "
        f"{test_metrics['accuracy']:.4f}"
        f" | balanced accuracy "
        f"{test_metrics['balanced_accuracy']:.4f}"
        f" | sensitivity "
        f"{test_metrics['sensitivity']:.4f}"
        f" | specificity "
        f"{test_metrics['specificity']:.4f}"
    )

    print(
        "Thời gian outer fold:",
        round(
            outer_elapsed_seconds / 60,
            2,
        ),
        "phút",
    )

    del final_model

    torch.cuda.empty_cache()


total_elapsed_seconds = (
    time.time() - experiment_start_time
)


print("\nĐã hoàn thành 4 outer folds.")
print(
    "Tổng thời gian:",
    round(
        total_elapsed_seconds / 60,
        2,
    ),
    "phút",
)

In [ ]:
inner_results = pd.DataFrame(
    inner_result_records
)

outer_results = pd.DataFrame(
    outer_result_records
)

predictions = (
    pd.DataFrame(prediction_records)
    .sort_values("sample_index")
    .reset_index(drop=True)
)


assert len(inner_results) == 12
assert len(outer_results) == 4
assert len(predictions) == 160
assert predictions["sample_index"].nunique() == 160

assert np.allclose(
    all_attention_weights.sum(axis=1),
    1.0,
    atol=1e-4,
)


metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "false_alarm_rate",
]


metric_summary = (
    outer_results[metric_columns]
    .agg(["mean", "std"])
    .T
    .reset_index()
    .rename(columns={
        "index": "metric",
    })
)


overall_metrics = calculate_metrics(
    predictions["true_label"],
    predictions["predicted_label"],
)


print("KẾT QUẢ THEO 4 OUTER FOLDS")

display(
    outer_results[
        [
            "test_subject",
            "inner_best_epochs",
            "final_epochs",
            "test_samples",
            *metric_columns,
            "fp",
            "fn",
            "elapsed_seconds",
        ]
    ].round(4)
)


print("\nTRUNG BÌNH 4 OUTER FOLDS")

display(
    metric_summary.round(4)
)


print("\nKẾT QUẢ GỘP 160 VIDEO")

for metric_name in metric_columns:
    print(
        f"{metric_name}:",
        round(
            overall_metrics[
                metric_name
            ],
            4,
        ),
    )


# Confusion matrix
matrix = confusion_matrix(
    predictions["true_label"],
    predictions["predicted_label"],
    labels=[0, 1],
)


confusion_display = (
    ConfusionMatrixDisplay(
        confusion_matrix=matrix,
        display_labels=[
            "ADL",
            "FALL",
        ],
    )
)

confusion_display.plot(
    cmap="Purples",
    values_format="d",
)

plt.title(
    "ME-STGCN-TA — Nested LOSO"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "me_stgcn_ta_confusion_matrix.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# Attention trung bình theo lớp
adl_attention = (
    all_attention_weights[
        labels == 0
    ].mean(axis=0)
)

fall_attention = (
    all_attention_weights[
        labels == 1
    ].mean(axis=0)
)

normalized_time = np.linspace(
    0.0,
    1.0,
    all_attention_weights.shape[1],
)


plt.figure(figsize=(10, 4))

plt.plot(
    normalized_time,
    adl_attention,
    marker="o",
    label="ADL",
)

plt.plot(
    normalized_time,
    fall_attention,
    marker="o",
    label="FALL",
)

plt.xlabel(
    "Thời gian chuẩn hóa của video"
)

plt.ylabel(
    "Attention weight trung bình"
)

plt.title(
    "Temporal Attention theo lớp"
)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "attention_by_class.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# Các video dự đoán sai
error_videos = predictions.loc[
    ~predictions["correct"]
].copy()


display(error_videos)

print(
    "Tổng video dự đoán sai:",
    len(error_videos),
)


# Lưu toàn bộ kết quả
inner_results.to_csv(
    OUTPUT_DIR
    / "inner_results.csv",
    index=False,
)

outer_results.to_csv(
    OUTPUT_DIR
    / "outer_results.csv",
    index=False,
)

predictions.to_csv(
    OUTPUT_DIR
    / "predictions.csv",
    index=False,
)

metric_summary.to_csv(
    OUTPUT_DIR
    / "metric_summary.csv",
    index=False,
)

error_videos.to_csv(
    OUTPUT_DIR
    / "error_videos.csv",
    index=False,
)


attention_table = pd.DataFrame(
    all_attention_weights,
    columns=[
        f"attention_{index:02d}"
        for index in range(
            all_attention_weights.shape[1]
        )
    ],
)

attention_table.insert(
    0,
    "video_id",
    video_ids,
)

attention_table.insert(
    1,
    "subject",
    subjects,
)

attention_table.insert(
    2,
    "label",
    labels,
)

attention_table.to_csv(
    OUTPUT_DIR
    / "attention_weights.csv",
    index=False,
)


print("\nĐã lưu kết quả:", OUTPUT_DIR)
print("Đã lưu models:", MODEL_DIR)